# 5장. 회귀 평가 지표

오차를 숫자 하나로 요약하면 어떤 정보가 빠지는가.

**준비:** 2장에서 쓴 서울 아파트 실거래가 CSV 네 개를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다.

## 5.1 여섯 가지 지표로 네 식 평가하기

In [ ]:
import os, re
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def period(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)
v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["집"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"] + " " + v["면적"].round().astype(int).astype(str)
v["연식"] = v["날짜"].dt.year - v["건축년도"]
v = v.sort_values("날짜").reset_index(drop=True)

train = v[v["날짜"].dt.year <= 2024].copy()
test = v[v["날짜"].dt.year == 2025].copy()
y = test["억"].to_numpy()
print(len(train), len(test), round(float(y.mean()), 2), round(float(np.median(y)), 2))

In [ ]:
GU, AGES = sorted(v["구"].unique()), [0, 10, 20, 30, 40, 50]
DONG = sorted(train["동네"].unique())

def onehot(s, cats):
    return (s.to_numpy()[:, None] == np.asarray(cats)[None, :]).astype(float)

def others(s):
    a = s.to_numpy()
    if len(a) == 1:
        return pd.Series([np.nan], index=s.index)
    return pd.Series([np.median(np.delete(a, i)) for i in range(len(a))], index=s.index)

of_dong, of_gu = train.groupby("동네")["억"].median(), train.groupby("구")["억"].median()
test["지난가"] = test["집"].map(train.groupby("집")["억"].median())
train["지난가"] = train.groupby("집")["억"].transform(others)

def ages(df):
    return onehot(df["연식"].clip(0, 59) // 10 * 10, AGES)

def past(df):
    filled = df["지난가"].fillna(df["동네"].map(of_dong)).fillna(df["구"].map(of_gu))
    return np.column_stack([np.log(filled), df["지난가"].isna().astype(float)])

MAKE = {
    "가": (lambda df: np.column_stack([df["면적"], df["층"], df["건축년도"], onehot(df["구"], GU)]), False),
    "나": (lambda df: np.column_stack([np.log(df["면적"]), df["층"], ages(df), onehot(df["구"], GU)[:, 1:]]), True),
    "다": (lambda df: np.column_stack([df["면적"], df["층"], ages(df), onehot(df["동네"], DONG)[:, 1:]]), False),
    "라": (lambda df: np.column_stack([np.log(df["면적"]), df["층"], ages(df), onehot(df["구"], GU)[:, 1:], past(df)]), True),
}
SHEET = {}
for name, (make, log) in MAKE.items():
    target = np.log(train["억"].to_numpy()) if log else train["억"].to_numpy()
    coef = np.linalg.lstsq(make(train), target, rcond=None)[0]
    SHEET[name] = np.exp(make(test) @ coef) if log else make(test) @ coef
print({k: s.round(2)[:3].tolist() for k, s in SHEET.items()}, y[:3].tolist())

In [ ]:
def marks(answer, actual):
    miss = answer - actual
    rate = np.abs(miss / actual)
    return {
        "RMSE": np.sqrt((miss ** 2).mean()),          # 억 원
        "MAE": np.abs(miss).mean(),                    # 억 원
        "MAPE": rate.mean() * 100,                     # %
        "MdAPE": np.median(rate) * 100,                # %
        "10% 이내": (rate <= 0.10).mean() * 100,       # %
        "평균 오차": miss.mean(),                      # 억 원
    }

card = pd.DataFrame({k: marks(s, y) for k, s in SHEET.items()}).T.round(2)
print(card.to_string())
rank = card.drop(columns="평균 오차").rank(ascending=True).assign(**{"10% 이내": card["10% 이내"].rank(ascending=False)}).astype(int)
print(rank.to_string())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# 그림 5-2
cols = list(rank.columns)
style = {"가": (GRAY, "-"), "나": (BRICK, "-"), "다": (TEAL, "-"), "라": (NAVY, "-")}
fig, ax = plt.subplots(figsize=(7, 3.6))
for name in rank.index:
    col = style[name][0]
    ax.plot(range(len(cols)), rank.loc[name], color=col, lw=2.2, marker="o", ms=9)
    for xi, c in enumerate(cols):
        ax.text(xi, rank.loc[name, c], name, ha="center", va="center", fontsize=7, color="white", weight="bold")
    ax.text(-0.25, rank.loc[name, cols[0]], f"식 {name}", ha="right", va="center", fontsize=8.5, color=col, weight="bold")
ax.set_xticks(range(len(cols)), [f"{c}\n({'클수록' if c == '10% 이내' else '작을수록'} 좋다)" for c in cols], fontsize=8)
ax.set_yticks([1, 2, 3, 4], ["1위", "2위", "3위", "4위"])
ax.set_ylim(4.5, 0.5)
ax.set_xlim(-0.8, len(cols) - 0.6)
ax.text(2.0, 3.5, "식 나와 식 다는 지표에 따라 2위와 3위가 바뀐다", ha="center", va="center", fontsize=8, color=NAVY, bbox=BOX)
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_2.png", dpi=300)
plt.show()

In [ ]:
print({k: int((s <= 0).sum()) for k, s in SHEET.items()})
worst = test.assign(예측=SHEET["다"].round(2)).sort_values("예측").head(3)
print(worst[["구", "동네", "면적", "층", "건축년도", "억", "예측"]].to_string(index=False))

## 5.2 제곱 오차와 절대 오차: 큰 오차의 영향과 지표가 선택하는 대푯값

In [ ]:
sq = (SHEET["라"] - y) ** 2
order = np.argsort(sq)[::-1]                       # 제곱 오차가 큰 거래부터
n = len(sq)
print(round(float(sq[order[0]] / sq.sum() * 100), 1), round(float(sq[order[: n // 1000]].sum() / sq.sum() * 100), 1),
      round(float(sq[order[: n // 100]].sum() / sq.sum() * 100), 1))
ab = np.abs(SHEET["라"] - y)
print(round(float(ab[order[0]] / ab.sum() * 100), 2), round(float(ab[order[: n // 1000]].sum() / ab.sum() * 100), 1),
      round(float(ab[order[: n // 100]].sum() / ab.sum() * 100), 1))

top = test.iloc[order[:3]].assign(예측=SHEET["라"][order[:3]].round(1))
print(top[["날짜", "동네", "단지명", "면적", "층", "억", "예측"]].to_string(index=False))

In [ ]:
# 그림 5-3
ab_sorted = np.sort(ab)[::-1]
share = np.arange(1, n + 1) / n * 100
cum_abs = np.cumsum(ab_sorted) / ab_sorted.sum() * 100
cum_sq = np.cumsum(ab_sorted ** 2) / (ab_sorted ** 2).sum() * 100
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(share, cum_sq, color=BRICK, lw=2.0, label="제곱 오차의 합에서 차지하는 비율 (RMSE)")
ax.plot(share, cum_abs, color=NAVY, lw=2.0, label="절대 오차의 합에서 차지하는 비율 (MAE)")
ax.plot([0, 100], [0, 100], color=GRAY, lw=0.8, ls=(0, (4, 3)))
k = n // 100
ax.scatter([1, 1], [cum_sq[k - 1], cum_abs[k - 1]], color=[BRICK, NAVY], s=28, zorder=3)
ax.annotate(f"오차가 큰 1%의 거래가\n제곱 오차의 {cum_sq[k - 1]:.1f}%", (1, cum_sq[k - 1]), (28, 44), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.annotate(f"같은 1%가 절대 오차의 {cum_abs[k - 1]:.1f}%", (1, cum_abs[k - 1]), (32, 12), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
ax.text(71, 64, "모든 거래의 오차가 같을 때", fontsize=7.5, color=GRAY, rotation=27)
ax.set_xlim(-1, 100); ax.set_ylim(0, 102)
ax.set_xlabel("오차가 큰 거래부터 센 비율(%)", fontsize=8)
ax.set_ylabel("오차의 합에서 차지하는 비율(%)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="lower right")
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_3.png", dpi=300)
plt.show()

In [ ]:
# 그림 5-4
fig, ax = plt.subplots(figsize=(7, 4.4))
ax.scatter(y, SHEET["라"], s=2, color=NAVY, alpha=0.25, lw=0)
ax.plot([0.5, 400], [0.5, 400], color=GRAY, lw=0.9)
ax.scatter(y[order[:3]], SHEET["라"][order[:3]], s=30, color=BRICK, zorder=3)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(0.5, 400); ax.set_ylim(0.5, 400)
ax.annotate("제곱 오차가 가장 큰 3건\n실제 175억~290억 원, 예측 18억~22억 원", (y[order[1]], SHEET["라"][order[1]]), (20, 1.6), fontsize=8, color=BRICK,
            arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.text(0.62, 150, "대각선보다 위: 예측이 실제보다 높다", fontsize=8, color=GRAY)
ax.text(30, 0.75, "대각선보다 아래: 예측이 실제보다 낮다", fontsize=8, color=GRAY)
ax.set_xticks([1, 3, 10, 30, 100, 300], ["1", "3", "10", "30", "100", "300"])
ax.set_yticks([1, 3, 10, 30, 100, 300], ["1", "3", "10", "30", "100", "300"])
ax.set_xlabel("실제 가격(억 원, 로그 눈금)", fontsize=8)
ax.set_ylabel("식 라의 예측(억 원, 로그 눈금)", fontsize=8)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_5_4.png", dpi=300)
plt.show()

In [ ]:
keep = np.ones(n, dtype=bool)
keep[order[0]] = False
print(pd.DataFrame({"74,046건": marks(SHEET["라"], y), "1건을 빼고": marks(SHEET["라"][keep], y[keep])}).T.round(2).to_string())

In [ ]:
print((card["RMSE"] / card["MAE"]).round(2).to_dict(), round(float(np.sqrt(np.pi / 2)), 3))

rows = []
for name, s in SHEET.items():
    miss = s - y
    rows.append((name, round(float(miss.mean()), 2), round(float(miss.std()), 2), round(float(np.sqrt(miss.mean() ** 2 + miss.var())), 2)))
print(pd.DataFrame(rows, columns=["식", "평균 오차", "오차의 표준편차", "√(평균² + 분산)"]).set_index("식").to_string())

In [ ]:
grid = np.arange(4, 20.01, 0.01)
RATE = {"RMSE": lambda c: np.sqrt(((c - y) ** 2).mean()), "MAE": lambda c: np.abs(c - y).mean(),
        "MAPE": lambda c: np.abs(c / y - 1).mean(), "MdAPE": lambda c: np.median(np.abs(c / y - 1)),
        "RMSLE": lambda c: np.sqrt(((np.log(c) - np.log(y)) ** 2).mean())}
CURVE = {name: np.array([f(c) for c in grid]) for name, f in RATE.items()}
best = {name: round(float(grid[np.argmin(curve)]), 2) for name, curve in CURVE.items()}
print(best)
print(round(float(y.mean()), 2), round(float(np.median(y)), 2), round(float(np.exp(np.log(y).mean())), 2))
print({k: round(float((y < c).mean() * 100), 1) for k, c in best.items()})

In [ ]:
# 그림 5-5
fig, ax = plt.subplots(figsize=(7, 4.2))
ax2 = ax.twinx()
ax2.hist(y, bins=np.arange(0, 40.5, 0.5), color=to_rgba(GRAY, 0.25), edgecolor="none")
ax2.set_yticks([]); ax2.spines[["top", "right", "left"]].set_visible(False)
ax2.set_ylim(0, 9000)
look = {"RMSE": (BRICK, "-"), "MAE": (NAVY, "-"), "MAPE": (TEAL, "-"), "MdAPE": (TEAL, (0, (4, 2))), "RMSLE": (GRAY, "-")}
note_ = {"RMSE": "평균", "MAE": "중앙값", "MAPE": "하위 25.8% 지점", "MdAPE": "하위 43.9% 지점", "RMSLE": "기하평균"}
for name, curve in CURVE.items():
    col, ls = look[name]
    ax.plot(grid, curve / curve.min(), color=col, lw=1.8, ls=ls, label=f"{name}: {best[name]:.2f}억 원 ({note_[name]})")
    ax.plot([best[name]] * 2, [0.9, 1], color=col, lw=0.8, ls=(0, (2, 2)))
    ax.scatter([best[name]], [1], color=col, s=34, zorder=3)
ax.legend(frameon=True, facecolor="white", edgecolor="none", framealpha=0.95, fontsize=8, loc="upper center", bbox_to_anchor=(0.37, 1.0), title="지표: 그 지표가 가장 작아지는 c", title_fontsize=8)
ax.set_ylim(0.9, 1.9); ax.set_xlim(4, 20)
ax.text(15.2, 0.925, "회색 막대: 2025년 거래 가격의 분포", fontsize=7.5, color="#6b7480")
ax.set_xlabel("모든 거래에 똑같이 답하는 가격 c(억 원)", fontsize=8)
ax.set_ylabel("지표의 값 ÷ 그 지표의 최솟값", fontsize=8)
ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_5.png", dpi=300)
plt.show()

In [ ]:
def rmsle(answer, actual):
    return float(np.sqrt(((np.log(answer) - np.log(actual)) ** 2).mean()))

print({k: (round(rmsle(s, y), 3) if (s > 0).all() else "계산 불가") for k, s in SHEET.items()})
one = np.array([10.0])
print([round(float(np.abs(a / one - 1)[0] * 100)) for a in (5.0, 20.0)], [round(abs(float(np.log(a / one)[0])), 3) for a in (5.0, 20.0)])

## 5.3 가격대별·구별 오차: 전체 지표에 가려지는 구간

In [ ]:
EDGES = [0, 5, 10, 15, 20, 30, 50, 400]
band = pd.cut(y, EDGES)

def by_band(answer):
    t = pd.DataFrame({"가격대": band, "치우침%": (answer / y - 1) * 100, "오차율": np.abs(answer / y - 1) * 100})
    return t.groupby("가격대", observed=True).agg(건수=("오차율", "size"), 치우침=("치우침%", "median"), 오차율=("오차율", "median")).round(1)

print(by_band(SHEET["라"]).to_string())
print(by_band(SHEET["가"])[["치우침", "오차율"]].T.to_string())

In [ ]:
# 그림 5-6
t = by_band(SHEET["라"])
labels = ["5억 이하", "5~10억", "10~15억", "15~20억", "20~30억", "30~50억", "50억 초과"]
x = np.arange(len(t))
fig, ax = plt.subplots(figsize=(7, 4.0))
ax.bar(x, t["오차율"], color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.9, width=0.62, label="오차율의 중앙값(MdAPE)")
ax.plot(x, t["치우침"], color=BRICK, marker="o", ms=5, lw=1.6, label="치우침의 중앙값(음수: 예측이 실제보다 낮다)")
for xi, (a, b) in enumerate(zip(t["오차율"], t["치우침"])):
    ax.text(xi, a + 0.9, f"{a:.1f}", ha="center", fontsize=8, color=NAVY)
    ax.text(xi, b - 3.4, f"{b:+.1f}", ha="center", fontsize=8, color=BRICK)
ax.axhline(0, color=GRAY, lw=0.8)
ax.axhline(card.loc["라", "MdAPE"], color=NAVY, lw=0.9, ls=(0, (4, 3)))
ax.text(-0.45, card.loc["라", "MdAPE"] + 0.9, f"전체 {card.loc['라', 'MdAPE']:.2f}%", ha="left", fontsize=8, color=NAVY)
ax.set_xticks(x, [f"{a}\n{c:,}건" for a, c in zip(labels, t["건수"])], fontsize=8)
ax.set_ylim(-31, 36)
ax.set_ylabel("%", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_5_6.png", dpi=300)
plt.show()

In [ ]:
t = pd.DataFrame({"구": test["구"].to_numpy(), "rate": np.abs(SHEET["라"] / y - 1) * 100, "miss": SHEET["라"] - y, "억": y})
g = t.groupby("구").agg(건수=("rate", "size"), 가격_중앙값=("억", "median"), 오차율=("rate", "median"),
                        RMSE=("miss", lambda s: np.sqrt((s ** 2).mean()))).round(2).sort_values("오차율")
print(g.iloc[[0, 1, 2, -3, -2, -1]].to_string())
print(round(float(np.corrcoef(g["가격_중앙값"], g["오차율"])[0, 1]), 2))

In [ ]:
# 그림 5-7
fig, ax = plt.subplots(figsize=(7, 5.6))
ypos = np.arange(len(g))[::-1]
whole = card.loc["라", "MdAPE"]
for yi, (name, row) in zip(ypos, g.iterrows()):
    col = BRICK if row["오차율"] > whole else NAVY
    ax.barh(yi, row["오차율"], height=0.66, color=to_rgba(col, 0.55 if col == BRICK else 0.22), edgecolor=col, lw=0.8)
    ax.text(row["오차율"] + 0.2, yi, f"{row['오차율']:.1f}%", va="center", fontsize=7.5, color=col, bbox=BOX)
    ax.text(23.6, yi, f"{row['가격_중앙값']:.1f}억", va="center", ha="right", fontsize=7.5, color="#6b7480")
ax.axvline(whole, color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.text(whole + 0.15, len(g) - 0.1, f"서울 전체 {whole:.2f}%", fontsize=8, color="#6b7480", va="bottom")
ax.text(23.6, len(g) - 0.1, "그 구의 가격 중앙값", fontsize=8, color="#6b7480", va="bottom", ha="right")
ax.set_yticks(ypos, g.index, fontsize=8)
ax.set_xlim(0, 24); ax.set_ylim(-0.7, len(g) + 0.9)
ax.set_xlabel("식 라의 구별 오차율 중앙값(%)", fontsize=8)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_5_7.png", dpi=300)
plt.show()

## 5.4 결정계수 R²: 기준선이 들어 있는 지표

In [ ]:
def r2(answer, actual, base=None):
    """1 - (식의 제곱 오차 합) / (기준 예측의 제곱 오차 합). base를 주지 않으면 actual의 평균이 기준이다."""
    base = actual.mean() if base is None else base
    return round(float(1 - ((answer - actual) ** 2).sum() / ((actual - base) ** 2).sum()), 3)

print({k: r2(s, y) for k, s in SHEET.items()})
spread = float(np.sqrt(((y - y.mean()) ** 2).mean()))
print(round(spread, 2), round(1 - (card.loc["라", "RMSE"] / spread) ** 2, 3))

m_train = float(train["억"].mean())
print(round(m_train, 2), round(float(y.mean()), 2), r2(np.full(n, m_train), y), {k: r2(s, y, m_train) for k, s in SHEET.items()})

In [ ]:
rows = []
for where in ["서울 전체", "강남구", "노원구"]:
    pick = np.ones(n, dtype=bool) if where == "서울 전체" else (test["구"] == where).to_numpy()
    spread = np.sqrt(((y[pick] - y[pick].mean()) ** 2).mean())
    for name in ["가", "라"]:
        mk = marks(SHEET[name][pick], y[pick])
        rows.append((where, name, int(pick.sum()), round(float(spread), 2), round(mk["RMSE"], 2), r2(SHEET[name][pick], y[pick]), round(mk["MdAPE"], 1)))
r2_table = pd.DataFrame(rows, columns=["범위", "식", "건수", "가격의 표준편차", "RMSE", "R²", "MdAPE"])
print(r2_table.to_string(index=False))

In [ ]:
# 그림 5-8
sub = r2_table[r2_table["식"] == "라"].reset_index(drop=True)
x = np.arange(len(sub))
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.bar(x - 0.2, sub["가격의 표준편차"], width=0.4, color=to_rgba(GRAY, 0.45), edgecolor=GRAY, lw=0.9, label="가격의 표준편차 (평균만 답할 때의 RMSE)")
ax.bar(x + 0.2, sub["RMSE"], width=0.4, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.9, label="식 라의 RMSE")
for xi, row in sub.iterrows():
    ax.text(xi - 0.2, row["가격의 표준편차"] + 0.35, f"{row['가격의 표준편차']:.2f}억", ha="center", fontsize=8, color="#6b7480")
    ax.text(xi + 0.2, row["RMSE"] + 0.35, f"{row['RMSE']:.2f}억", ha="center", fontsize=8, color=NAVY)
    ax.text(xi, -3.4, f"R² = {row['R²']:.3f}\nMdAPE = {row['MdAPE']:.1f}%", ha="center", va="top", fontsize=8.5, color=BRICK, linespacing=1.5)
ax.set_xticks(x, [f"{a} ({c:,}건)" for a, c in zip(sub["범위"], sub["건수"])], fontsize=8.5)
ax.set_ylim(0, 21)
ax.set_ylabel("억 원", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper right")
frame(ax)
fig.subplots_adjust(bottom=0.27, left=0.09, right=0.98, top=0.96)
fig.savefig("fig_5_8.png", dpi=300)
plt.show()